In [3]:
from src.runtime_constants import rr
from src.db.mysql import DbConnection
from src import external_api

sheet = external_api.read_sheet("1TIu0NXPH8LFgVmlzA4-y2GdI80MbnQW_-FAvAxDCvys", "loved audience", True)

bulk_data = []
for row in sheet:
    if row["tag_id_int"] == "":
        continue
    tag_id_int = int(row["tag_id_int"])
    loved_ratio = float(str(row["loved_ratio"]).replace(",", "."))
    bulk_data.append((tag_id_int, loved_ratio))

if len(bulk_data) > 0:
    rr.set_env(rr.ENV_PROD, True)
    conn = DbConnection()
    conn.query("""
    DELETE FROM platform.ext_loved_ratio
    """)
    conn.bulk("""
    INSERT INTO platform.ext_loved_ratio (tag_id_int, loved_ratio) VALUES (%s, %s)
    """, bulk_data)
    conn.commit()
    conn.close()
    print("DONE")

DONE


In [15]:
from src.runtime_constants import rr
from src.db.mysql import DbConnection
from src import external_api

sheet = external_api.read_sheet("1TIu0NXPH8LFgVmlzA4-y2GdI80MbnQW_-FAvAxDCvys", "loved audience", True)

query = """
SELECT z1.tag_id_int, z1.loyalty_installs, ROUND(AVG(z1.rejected_cnt)) / 1988 AS rejected_median,
z1.rejected_min / 1988 AS rejected_min, z1.rejected_max / 1988 AS rejected_max
FROM (
SELECT aa.id AS tag_id_int, r.rejected_cnt,
MAX(r.rejected_cnt) OVER (PARTITION BY aa.id) AS rejected_max,
MIN(r.rejected_cnt) OVER (PARTITION BY aa.id) AS rejected_min,
COUNT(1) OVER (PARTITION BY aa.id) AS total_num,
ROW_NUMBER() OVER (PARTITION BY aa.id ORDER BY r.rejected_cnt) AS row_num,
aa.loyalty_installs
  FROM platform.audience_angle aa,
       platform.audience_angle_rejected_per_app r,
       app_temp_data.erik_top_500 e
 WHERE r.id = aa.id
   AND aa.angle_cnt = 1
   AND e.app_id_int = r.app_id_int
) z1
WHERE z1.row_num IN (FLOOR((z1.total_num + 1) / 2), FLOOR((z1.total_num + 2) / 2) )
GROUP BY z1.tag_id_int, z1.rejected_max, z1.rejected_min, z1.loyalty_installs;
"""
conn = DbConnection(rr.ENV_PROD, True)
rows = conn.select_all(query)
rejected_map = {}
for row in rows:
    rejected_map[row["tag_id_int"]] = row
conn.close()

values = []
arr = []
for k in sheet[0]:
    arr.append(k)
# arr.append("rejected_median")
# arr.append("rejected_min")
# arr.append("rejected_max")
# arr.append("loyalty_installs")
values.append(arr)

for row in sheet:
    tag_id_int = int(row["tag_id_int"])
    # loyalty_installs = 0
    # rejected_median = 0
    # rejected_min = 0
    # rejected_max = 0
    if tag_id_int in rejected_map:
        row["rejected_median"] = float(rejected_map[tag_id_int]["rejected_median"])
    #     rejected_median = float(rejected_map[tag_id_int]["rejected_median"])
    #     rejected_min = float(rejected_map[tag_id_int]["rejected_min"])
    #     rejected_max = float(rejected_map[tag_id_int]["rejected_max"])

    arr = []
    for k in row:
        arr.append(row[k])
    # arr.append(rejected_median)
    # arr.append(rejected_min)
    # arr.append(rejected_max)
    #arr.append(loyalty_installs)
    values.append(arr)

values
external_api.write_sheet("1TIu0NXPH8LFgVmlzA4-y2GdI80MbnQW_-FAvAxDCvys", "loved audience", values)


{'spreadsheetId': '1TIu0NXPH8LFgVmlzA4-y2GdI80MbnQW_-FAvAxDCvys',
 'updatedRange': "'loved audience'!A1:N237",
 'updatedRows': 237,
 'updatedColumns': 14,
 'updatedCells': 3318}